# **Máster en Behavioral Data Science**
## **Instituto de Formación Continua (IL3) - Universitat de Barcelona**
## **Módulo 9: Aprendizaje Profundo - Reto 2** - (Notebook 1/4)
Autor: **Meysam Madadi**

Colaborador: **Julio C. S. Jacques Junior**

---

# **Reto 2:** Predicción de personalidad aparente usando aprendizaje profundo y datos multimodales

En esta práctica, construiremos diferentes modelos para **predecir la personalidad aparente** de los individuos a partir de múltiples tipos de datos. Es decir, las etiquetas (valores de personalidad) fueron anotadas por observadores externos. Nuestro primer modelo utilizará únicamente **información textual** obtenida de las transcripciones de los videos de entrada. Nuestro segundo modelo utilizará **información visual** (algunas imágenes para cada entrada de video). Finalmente, combinaremos texto e información visual en un **enfoque multimodal**.


# **Objetivo general**
- Manejar datos multimodales.
- Desarrollar una rede multimodal utilizando una estrategia de fusión sencilla.
- Comparar diferentes enfoques, basados en texto, imágenes y una combinación de ellos para la predicción de la personalidad.








# **Los objetivos de este Jupyter notebook**
- Conocer el conjunto de datos y visualizarlo.
- Aprender a extraer características de los datos de texto.

## Nota IMPORTANTE
- En este *Jupyter Notebook*, preprocesamos los datos textuales (transcripciones de videos) para **extraer características del lenguaje natural**. Primero, analizamos el texto y lo dividimos en oraciones. Luego, cada oración se descompone en sus componentes (palabras y símbolos), los cuales se utilizan como entrada para el algoritmo de extracción de características.
- Como veréis, analizar información de texto y preprocesarla puede ser un poco complejo. El objetivo de este Jupyter Notebook es proporcionar el código y la explicación para el ejemplo/práctica dado, y **no profundizar en los detalles del preprocesamiento**, ya que se podrían usar diferentes estrategias.


# **Conjunto de datos**
- Usamos un subconjunto del conjunto de datos [**First Impressions**](https://chalearnlap.cvc.uab.cat/dataset/24/description/) para este reto. Está compuesto por 6000 clips (con una duración promedio de 15 segundos cada uno) extraídos de más de 3000 videos de alta definición, de personas frente a una cámara y hablando en inglés. Los videos se dividen en conjuntos de entrenamiento, validación y prueba con una proporción de 4:1:1. Las personas en los videos muestran diferentes géneros, edades y etnias.
- Los videos están etiquetados con variables de rasgos de personalidad. Los rasgos de personalidad considerados fueron los del Modelo de “Five Factor” (también conocido como los “**Big-Five**”). Este modelo describe la personalidad humana a lo largo de cinco dimensiones: **“Openness”**, **“Conscientiousness”**, **“Extraversion”**, **“Agreeableness”**, y **“Neuroticism”**. En este conjunto de datos, “Neuroticism” fue anotado como “Emotional stability”, es decir, lo opuesto al neuroticismo. Cada clip tiene etiquetas para estos cinco rasgos representados con un valor dentro del rango [0, 1]. Las transcripciones de los videos también se proporcionan en el conjunto de datos como modalidad adicional a los videos.
- **Para poder trabajar con los datos en un tiempo razonable en Colab, preprocesamos los datos**. Es decir, se extrajeron 4 imágenes RGB de cada videoclip, igualmente espaciadas. También procesamos las transcripciones de video usando el modelo [BERT](https://pypi.org/project/pytorch-pretrained-bert/) y extraemos características por palabra. El código de esta parte está disponible en este Jupyter Notebook como material complementario.

- Los datos (información visual y transcripciones) preprocesados se descargan desde nuestro servidor en la siguiente celda. Las transcripciones se han guardado como un diccionario en el que las claves son los nombres de los videos. A continuación, visualizamos una muestra como ejemplo de datos.

In [ ]:
# Download and unzip the data
!wget https://data.chalearnlap.cvc.uab.cat/Colab_MFPDS/2024BehaviorDSMaster/M9_r2/data_final.zip
!unzip ./data_final.zip

# Download video transcriptions
!wget https://data.chalearnlap.cvc.uab.cat/Colab_MFPDS/2024BehaviorDSMaster/M9_r2/train-transcription.zip
!unzip train-transcription.zip

In [ ]:
# Visualize a sample data
# You can change "vid_name" to another valid filenames (from train set) to visualize other samples

from glob import glob
import cv2
import numpy as np
import pickle
from matplotlib import pyplot as plt

root = './data_final/train/'
vid_name = '2oEUp9sGRGk.000'

fig, (ax1, ax2, ax3, ax4) = plt.subplots(1, 4, figsize=(15, 4))
fig.suptitle('Sampled images from the video', fontsize=14, fontweight='bold')

# reading and showing images
files = sorted(glob(root+vid_name+'/*.jpg'))
for i in range(4):
  img = cv2.imread(files[i])
  img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
  eval('ax'+str(i+1)).imshow(img)

# reading and printing the transcription
trans = pickle.load(open('transcription_training.pkl','rb'))
print('\n\n\nTranscription is:')
print(trans[vid_name+'.mp4'])

# reading and printing the personality labels
traits = np.load(open(root+vid_name+'/annotation.npy', 'rb'))
print('\nApparent personality labels are:')
print('extraversion\tneuroticism\tagreeableness\tconscientiousness\topenness')
print('{0:0.2f}\t\t{1:0.2f}\t\t{2:0.2f}\t\t{3:0.2f}\t\t\t{4:0.2f}\t\t'.format(traits[0],traits[1],traits[2],traits[3],traits[4]))

# **Procesamiento de Lenguaje Natural (*NLP*): Extracción de características de texto**

Existen diferentes maneras de extraer características de texto. Por ejemplo: 1) ["bag of words"](https://machinelearningmastery.com/gentle-introduction-bag-words-model/), 2) ["word2vec"](https://www.kdnuggets.com/2021/11/guide-word-embedding-techniques-nlp.html) y 3) redes ["Transformers"](https://www.kdnuggets.com/2021/11/guide-word-embedding-techniques-nlp.html) basadas en aprendizaje profundo. "Bag of words" crea vectores enormes y no densos, haciendo que las mediciones de similitud sean una tarea difícil. Además, los vectores están predefinidos y no tienen en cuenta el contexto de la oración. "Word2vec" resuelve algunos problemas de "bag of words". Sin embargo, una palabra con significados diferentes puede tener la misma representación ("*embeddings*"). En este Reto usaremos la tercera opción, que no tiene estos problemas. El primer modelo de este tipo se llama BERT. Usaremos un modelo BERT preentrenado para extraer representaciones de palabras ("*embeddings*"), como se describe a continuación.

## Instalando la biblioteca BERT

Primero, instalamos la biblioteca [pytorch-pretrained-bert](https://pypi.org/project/pytorch-pretrained-bert/), que contiene varios modelos de lenguaje preentrenado, incluidos BERT y GPT.


In [ ]:
!pip install pytorch-pretrained-bert

# **Preprocesamiento de datos ("*parsing*"):**

## Analizando y preprocesando las transcripciones

- A continuación, definimos una función básica para analizar y dividir las transcripciones en oraciones. Para esto, utilizamos delimitadores simples para encontrar el final de una oración: "**?**", "**!**" y "**.**". Las transcripciones pueden no tener un delimitador al final o pueden comenzar o terminar con "**...**". Estos casos también se manejan en la función definida.
- Para mostrar al BERT el **inicio/final** de una oración, también se agregan "símbolos especiales" a las oraciones, "**[CLS]**" y "**[SEP]**".
- Un ejemplo de cómo un archivo de transcripción se divide en oraciones se muestra al final de esta celda (después de ejecutarla).

In [ ]:
# Split the text into sentences
def split(text):
  indices = []
  for delimiter in ['?','!','.']:
    # Start searching for the substring from the beginning of the string
    start_index = 0
    # Continue searching until the substring is not found in the remaining part of the string
    while True:
        # Find the next occurrence of the substring starting from the current start_index
        index = text.find(delimiter, start_index)
        if index == -1:
            # If the substring is not found, break out of the loop
            break
        else:
            # If the substring is found, add its start index to the list of indices
            indices.append(index)
            # Update the start index to start searching for the next occurrence of the substring
            start_index = index + 1

  # Remove indices with '...' delimiter that detected above
  indices2remove = []
  # Start searching for the substring from the beginning of the string
  start_index = 0
  # Continue searching until the substring is not found in the remaining part of the string
  while True:
      # Find the next occurrence of the substring starting from the current start_index
      index = text.find('...', start_index)
      if index == -1:
          # If the substring is not found, break out of the loop
          break
      else:
          # If the substring is found, add its start index to the list of indices
          indices2remove.append(index)
          indices2remove.append(index+1)
          indices2remove.append(index+2)
          # Update the start index to start searching for the next occurrence of the substring
          start_index = index + 3
  for i in indices2remove:
    indices.remove(i)

  if len(indices) == 0: return ["[CLS] " + text + " [SEP]"]

  # Sort the indices
  indices = sorted(indices)
  indices = [-1]+indices if indices[-1]==len(text)-1 else [-1]+indices+[len(text)]

  # Start spliting the sentences
  # [CLS] and [SEP] are required to identify the begin and end.
  sentences = []
  for i in range(len(indices)-1):
    sentences.append("[CLS] " + text[indices[i]+1:indices[i+1]+1] + " [SEP]")

  return sentences


###########################################
# An example of how the sentences are splitted
###########################################
text = "Who was Jim Henson? Jim Henson was a puppeteer."
text = split(text)
print(text)

## Cargando el modelo pre-entrenado y los datos

- BERT tiene varios modelos preentrenados, que se pueden categorizar según el idioma 1) idioma (inglés, chino y multilingüe), 2) el uso de mayúsculas y minúsculas (con o sin mayúsculas) y 3) el tamaño del modelo (con 12 capas o grande con 24 capas). En este Reto, usaremos el modelo "**bert-base-uncased**" que está entrenado para oraciones en inglés en minúsculas ("*uncased*").
- La entrada del modelo será una secuencia de palabras y símbolos. El proceso de convertir una oración en sus componentes se llama "**tokenization**" y también lo maneja la biblioteca.


In [ ]:
import torch
from pytorch_pretrained_bert import BertTokenizer, BertModel
import pickle
import numpy as np

# Load pre-trained model tokenizer (vocabulary)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Load pre-trained model (weights)
model = BertModel.from_pretrained('bert-base-uncased')
model.eval()
# If GPU is available, put everything on cuda to speed up feature extraction
if torch.cuda.is_available():
  model.to('cuda')

# Load transcriptions into a dictionary
# Keys are video names
trans = pickle.load(open('transcription_training.pkl','rb'))
keys = trans.keys()


## Extracción de características

- A continuación, extraemos características de texto utilizando el modelo BERT, utilizando un bucle para iterar sobre cada transcripción de video, y otro para iterar sobre cada oración de la transcripción.
- La entrada a BERT es una secuencia de texto "*tokenizado*".
- Como salida, se toma la representación ("*embedding*") de la última capa: **un vector de tamaño 768 para cada "*token*"**.
- **Al final, se apilan las características de todos los "*tokens*" para cada transcripción de video.**
- NOTA: cada transcripción tiene un número diferente de palabras y símbolos ("*tokens*") y, por lo tanto, **el vector de características de salida de cada muestra de dato tiene una longitud variable** entre todas las muestras del conjunto de datos.


In [ ]:
# A loop over video transcriptions
for i, k in enumerate(keys):

  # Note that "split" is the funcion we defined before
  text = split(trans[k])

  features = []
  # A loop over sentences
  for s in text:
    print(str(i)+" "+k+": "+s)
    # Tokenized input
    tokenized_text = tokenizer.tokenize(s)

    # Convert token to vocabulary indices
    indexed_tokens = tokenizer.convert_tokens_to_ids(tokenized_text)

    # Convert inputs to PyTorch tensors
    tokens_tensor = torch.tensor([indexed_tokens])

    # If GPU is available, put everything on cuda to speed up feature extraction
    if torch.cuda.is_available():
      tokens_tensor = tokens_tensor.to('cuda')

    # Predict hidden states features for each layer
    with torch.no_grad():
        encoded_layers, _ = model(tokens_tensor, output_all_encoded_layers=False)

    # Accumulate encoded layers for all sentences
    # Discard [CLS] and [SEP] tokens
    features.append(encoded_layers.cpu().numpy()[:, 1:-1, :])

    # printing the feature size of each transcript (for debug purpose)
    aux = features[-1]
    print("Feature size (sentence):", np.array(aux).shape)

  # stack all sentences
  features = np.concatenate(features, axis=1)

  print("Feature size (transcript):", features.shape)
  print("----")

  # The processed features (of each transcript) are already saved in our server
  # and will be loadead later by the other notebook files
  # This process of feature extraction is shown for illustration purpose only.